# ggblab Examples

This notebook demonstrates basic usage of `ggblab`. It walks through loading a `.ggb` file, displaying it in the GeoGebra widget, and parsing the construction for inspection and manipulation.

Key points:

- Large constructions are handled using `getBase64` / `setBase64` (zip + base64).
- Run cells from top to bottom for the examples to work as intended.
- Cells that produce output (e.g., version checks, object lists) will display their results directly below the code cell.


In [1]:
# this magic for develop only
%load_ext autoreload
%autoreload 2

## Create and initialize GeoGebra

The next two cells import the `GeoGebra` controller and initialize the widget. The import cell does not print output; the initialization cell may open the widget and produce transient console output in the notebook UI.

## Create GeoGebra controller

Import and construct the `GeoGebra` controller; the next cell initializes the widget in the JupyterLab UI.

In [2]:
from ggblab import GeoGebra

## Initialize the GeoGebra widget

This cell runs `await GeoGebra().init()` to open the GeoGebra widget attached to the current kernel. Expect the widget to appear in the JupyterLab UI; the cell may print status messages or the kernel may emit frontend messages (these appear in the notebook output area).

In [136]:
# open GeoGebra Widget on left-side
ggb = await GeoGebra().init()

Using local cached file: xsd/common.xsd


## Version checks (package)

This cell prints the installed `ggblab` package version. Expect a short version string as the cell output.

In [137]:
# ggblab version
import ggblab
ggblab.__version__

'1.1.0'

## Version checks (applet)

This cell queries the GeoGebra applet with `getVersion()` and prints the applet version string. The output helps verify compatibility.

In [138]:
# GeoGebra Applet version
r = await ggb.function("getVersion")
r

'5.2.909.9'

## Load a .ggb file

Load a `.ggb` file from disk into the local `ggb.construction` object for inspection and sending to the applet.

In [139]:
# load from .ggb (zipped, and base64 encoded)
c = ggb.file.load('2025_06_08.ggb')

## Send construction to the applet

Use `setBase64` to send the loaded .ggb (zip+base64) to the GeoGebra view so it is rendered in the widget.

In [140]:
# sending loaded construction to GeoGebra view and draw
r = await ggb.function("setBase64", [ggb.construction.base64_buffer.decode('utf-8')])

## Parse the construction XML

Decode the construction XML into Python structures (dicts/lists) so you can inspect and modify element attributes programmatically.

In [8]:
# parse loaded xml as python dict
o = c.ggb_schema.decode(c.geogebra_xml)

In [9]:
# list of all object names
[e['@label'] for e in o['element']]

['O_{2}',
 'A',
 'B',
 'c',
 'n',
 "O'",
 'p',
 'G',
 'F_{2}',
 'j',
 'l',
 'β',
 'γ',
 't1',
 'a',
 'c_1',
 'g_1',
 't2',
 'a_1',
 'c_2',
 'i_1',
 'k',
 'm',
 'δ',
 'ε',
 'q',
 'r',
 'e',
 'A_{1}',
 'C_{1}',
 'H',
 'J',
 'K',
 'ζ',
 'η',
 'O_{1}',
 'θ',
 's',
 'b',
 "P'",
 'P',
 't',
 'F_{1}',
 'h_1',
 'j_1',
 'k_1',
 'l_1',
 'm_1',
 'O_3',
 'p_1',
 'n_1',
 'C',
 'q_1',
 'r_1',
 'Q',
 'R',
 'D',
 't3',
 'd',
 'o',
 'l_2',
 'α',
 'O',
 'f',
 'i',
 't4',
 'c_3',
 'i_2',
 'a_2',
 'h',
 'g',
 'f_1',
 's_1',
 'd_1',
 'e_1',
 'E',
 'F',
 'f_2',
 'd_2',
 'L']

In [10]:
[e['@label'] for e in o['element']].index('A')

1

In [11]:
o['element'][1]

{'@type': 'point',
 '@label': 'A',
 'show': [{'@object': True, '@label': True, '@ev': 4}],
 'objColor': [{'@r': 176, '@g': 0, '@b': 32, '@alpha': 0.0}],
 'layer': [{'@val': 9}],
 'labelMode': [{'@val': 0}],
 'animation': [{'@step': '0.1', '@type': 1, '@playing': False}],
 'pointSize': [{'@val': 5.0}],
 'pointStyle': [{'@val': 0}],
 'coords': [{'@x': '10', '@y': '0', '@z': '1'}]}

In [12]:
# change visible state of object 'A'
o['element'][1]['show'][0]['@object'] = False

In [13]:
import xmlschema

In [14]:
# construct attributes from dict
x = xmlschema.etree_tostring(c.ggb_schema.encode(o['element'][1], 'element'))
print(x)

<element type="point" label="A">
    <show object="false" label="true" ev="4" />
    <objColor r="176" g="0" b="32" alpha="0.0" />
    <layer val="9" />
    <labelMode val="0" />
    <animation step="0.1" type="1" playing="false" />
    <pointSize val="5.0" />
    <pointStyle val="0" />
    <coords x="10" y="0" z="1" />
</element>


## Temporary updates with `preserve()`

The `preserve()` context manager snapshots the applet state (using a zip+base64 snapshot) and restores it automatically when the block exits.

Usage notes:

- The snapshot is taken with `getBase64` and restored with `setBase64` when available.
- The yielded `snap` object includes `base64_zip`, `xml` (optional), `timestamp`, `size_bytes`, and `sha1`.
- Call `await snap.restore()` to restore immediately; call `snap.release()` to drop the stored snapshot from memory.

In [15]:
import asyncio

In [16]:
# update the applet
async with ggb.preserve() as snap:
    r = await ggb.function("evalXML", [x])
    await asyncio.sleep(3)

## Advanced inspection

The following examples show how to decode the base64 snapshot, open the embedded zip, and inspect internal files (including the main construction XML). Expected outputs include a list of filenames inside the `.ggb` zip and the decoded XML string(s).

## Inspect base64 snapshot

Decode the base64 snapshot and inspect the zipped content (files inside the .ggb). This is useful when debugging or extracting the main construction XML.

In [17]:
import base64
import zipfile
import io

In [18]:
type(snap.base64_zip)

str

In [19]:
with zipfile.ZipFile(io.BytesIO(base64.b64decode(snap.base64_zip))) as zf:
    for info in zf.infolist():
        if info.filename == 'geogebra.xml':
            with zf.open(info) as f:
                xml = f.read()

In [20]:
type(xml.decode())

str

In [21]:
# r = await ggb.function("setXML", [xml.decode()])

In [22]:
await ggb.function("setBase64", [snap.base64_zip])

## draw step-by-step from new construction

In [135]:
# draw step-by-step from new construction
r = await ggb.function("newConstruction")

names = [e['@label'] for e in o['element']]
for n in names:
    cmd = None
    ci, co = None, None
    for _c in o['command']:
        _ci = tuple(zip(*_c['input'].items()))[1]
        _co = tuple(zip(*_c['output'].items()))[1]
        if n in _co:
            cmd = _c
            ci = _ci
            co = _co
            break

    match names.index(n):
        case int(i):
            ei = o['element'][i]
            if cmd:
                if co.index(n) == 0:
                    x = xmlschema.etree_tostring(c.ggb_schema.encode(cmd, 'construction/command'))
                    r = await ggb.function("evalXML", [x])
                    # print(f"command: {n}")
            x = xmlschema.etree_tostring(c.ggb_schema.encode(ei, 'element'))
            r = await ggb.function("evalXML", [x])
            # print(f"element:{n}")

In [23]:
ggb.comm.logs

['register_target_cb: <ipykernel.comm.comm.Comm object at 0x11a854910>',
 'Clients connected: 1 (connects+=1, disconnects+=0)',
 'Clients connected: 1 (connects+=2, disconnects+=2)',
 'Clients connected: 1 (connects+=84, disconnects+=84)',
 'Clients connected: 1 (connects+=35, disconnects+=35)',
 'Clients connected: 1 (connects+=72, disconnects+=72)',
 'Clients connected: 1 (connects+=88, disconnects+=88)',
 'Clients connected: 1 (connects+=83, disconnects+=83)',
 'Clients connected: 1 (connects+=81, disconnects+=81)']

In [24]:
ggb.comm.recv_msgs

{}

In [25]:
ggb.comm.pending_futures

{}

In [26]:
ggb.comm.recv_events.queue

deque([{'type': 'start', 'payload': {}},
       {'type': 'add', 'payload': 'O_{2}'},
       {'type': 'add', 'payload': 'A'},
       {'type': 'add', 'payload': 'B'},
       {'type': 'add', 'payload': 'c'},
       {'type': 'add', 'payload': 'n'},
       {'type': 'add', 'payload': "O'"},
       {'type': 'add', 'payload': 'p'},
       {'type': 'add', 'payload': 'G'},
       {'type': 'add', 'payload': 'F_{2}'},
       {'type': 'add', 'payload': 'j'},
       {'type': 'add', 'payload': 'l'},
       {'type': 'add', 'payload': 'β'},
       {'type': 'add', 'payload': 'γ'},
       {'type': 'add', 'payload': 't1'},
       {'type': 'add', 'payload': 'a'},
       {'type': 'add', 'payload': 'c_1'},
       {'type': 'add', 'payload': 'g_1'},
       {'type': 'add', 'payload': 't2'},
       {'type': 'add', 'payload': 'a_1'},
       {'type': 'add', 'payload': 'c_2'},
       {'type': 'add', 'payload': 'i_1'},
       {'type': 'add', 'payload': 'k'},
       {'type': 'add', 'payload': 'm'},
       {'type': 'a

## ggblab_extra — construction protocol from the applet

These cells demonstrate how to extract a construction protocol (objects, commands, and dependency metadata) from a running GeoGebra applet using `ggblab_extra`.

They show how to initialize a structured `DataFrame` from the applet (`ConstructionIO.initialize_dataframe(ggb, use_applet=True)`) so you can analyze element attributes, sequences, and dependencies programmatically. These are analysis helpers and do not alter the stored applet state unless you explicitly call functions that modify the applet.

In [54]:
import polars as pl
import networkx as nx

In [55]:
from ggblab_extra.construction_io import ConstructionIO
df = await ConstructionIO.initialize_dataframe(ggb, use_applet=True)

In [56]:
df

Sequence,Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
u32,str,str,str,str,str,u32,bool,bool,bool
0,"""O_{2}""","""point""",null,"""O_{2} = (-2.1, -0.1)""",null,9,true,true,false
1,"""A""","""point""",null,"""A = (10, 0)""",null,2,true,true,false
2,"""B""","""point""","""Point(Line(O_{2}, A))""","""B = (-9.6, -0.2)""",null,9,true,false,false
3,"""c""","""circle""","""Circle(O_{2}, B)""","""c: (x + 2.1)² + (y + 0.1)² = 5…",null,9,true,false,false
4,"""n""","""line""","""Line(O_{2}, A)""","""n: y = -0.1""",null,3,false,false,false
5,"""O'""","""point""","""Midpoint(O_{2}, A)""","""O' = (3.9, 0)""",null,3,false,true,false
6,"""p""","""circle""","""Circle(O', A)""","""p: (x - 3.9)² + (y + 0)² = 36.…",null,3,false,false,false
7,"""G""","""point""","""Intersect(c, p)""","""G = (2.4, 5.8)""",null,1,false,false,false
8,"""F_{2}""","""point""","""Intersect(c, p)""","""F_{2} = (2.5, -5.9)""",null,1,true,true,false


## ggblab_extra — construction tree parser

Use `ConstructionTreeParser` to build a graph of construction dependencies and perform graph analyses. The examples below show how to:

- Create the parser with `p = ConstructionTreeParser(df)`
- Build the full dependency graph with `g1 = p.parse()`
- Extract bounded subgraphs with `g2 = p.parse_subgraph(max_depth=3)`

These operations are for inspection and analysis; they help you visualize and trace construction dependency paths.

In [57]:
from ggblab_extra.construction_parser import ConstructionTreeParser

In [58]:
p = ConstructionTreeParser(df)

In [59]:
g1 = p.parse()

In [60]:
labels_map = {}
for n, t in p.df["Name", "Type"].rows():
    g1.nodes[n]["label"] = f"{n} ({t})"

In [61]:
nx.write_network_text(g1)

╟── O_{2} (point)
╎   ├─╼ B (point) ╾ A (point)
╎   │   ├─╼ c (circle) ╾ O_{2} (point)
╎   │   │   ├─╼ G (point) ╾ p (circle)
╎   │   │   │   ├─╼ β (angle) ╾ A (point), O_{2} (point)
╎   │   │   │   ├─╼ t1 (triangle) ╾ A (point), O_{2} (point)
╎   │   │   │   │   ├─╼ a (segment) ╾ A (point), G (point)
╎   │   │   │   │   ├─╼ c_1 (segment) ╾ G (point), O_{2} (point)
╎   │   │   │   │   └─╼ g_1 (segment) ╾ O_{2} (point), A (point)
╎   │   │   │   ├─╼ k (segment) ╾ A (point)
╎   │   │   │   ├─╼ δ (angle) ╾ O_{2} (point), A (point)
╎   │   │   │   ├─╼ f_1 (segment) ╾ A (point)
╎   │   │   │   └─╼  ...
╎   │   │   ├─╼ F_{2} (point) ╾ p (circle)
╎   │   │   │   ├─╼ γ (angle) ╾ A (point), O_{2} (point)
╎   │   │   │   ├─╼ t2 (triangle) ╾ A (point), O_{2} (point)
╎   │   │   │   │   ├─╼ a_1 (segment) ╾ A (point), F_{2} (point)
╎   │   │   │   │   ├─╼ c_2 (segment) ╾ F_{2} (point), O_{2} (point)
╎   │   │   │   │   └─╼ i_1 (segment) ╾ O_{2} (point), A (point)
╎   │   │   │   ├─╼ m (segment) ╾ A

In [62]:
g2 = p.parse_subgraph(max_depth=3)

parse_subgraph: timeout exceeded during candidate evaluation (10.00s), falling back to parse_subgraph_legacy


In [63]:
nx.write_network_text(g2)

╟── O_{2}
╎   ├─╼ B ╾ A
╎   │   ├─╼ e
╎   │   └─╼ c
╎   │       ├─╼ l
╎   │       │   └─╼ r ╾ j
╎   │       │       └─╼ O_{1}
╎   │       │           ├─╼ t
╎   │       │           │   └─╼ F_{1}
╎   │       │           │       ├─╼ h_1
╎   │       │           │       │   ├─╼ p_2
╎   │       │           │       │   │   ├─╼ q_2
╎   │       │           │       │   │   │   └─╼ N
╎   │       │           │       │   │   └─╼ r_2
╎   │       │           │       │   ├─╼ k_2
╎   │       │           │       │   │   ├─╼ m_2
╎   │       │           │       │   │   └─╼ n_2
╎   │       │           │       │   ├─╼ k_1
╎   │       │           │       │   └─╼ j_1
╎   │       │           │       └─╼ t_1
╎   │       │           │           └─╼ C'
╎   │       │           ├─╼ b
╎   │       │           │   └─╼ P'
╎   │       │           └─╼ O_3
╎   │       │               └─╼ p_1
╎   │       │                   ├─╼ C
╎   │       │                   ├─╼ E
╎   │       │                   └─╼ D
╎   │       ├─╼ j


In [64]:
p.df.filter(~pl.col("Auxiliary"))["Sequence", "Type", "Name", "Command", "Value", "Layer", "DependsOn"]

Sequence,Type,Name,Command,Value,Layer,DependsOn
u32,str,str,str,str,u32,list[str]
0,"""point""","""O_{2}""",null,"""O_{2} = (-2.1, -0.1)""",9,[]
1,"""point""","""A""",null,"""A = (10, 0)""",2,[]
2,"""point""","""B""","""Point(Line(O_{2}, A))""","""B = (-9.6, -0.2)""",9,"[""A"", ""O_{2}""]"
3,"""circle""","""c""","""Circle(O_{2}, B)""","""c: (x + 2.1)² + (y + 0.1)² = 5…",9,"[""A"", ""B"", ""O_{2}""]"
4,"""line""","""n""","""Line(O_{2}, A)""","""n: y = -0.1""",3,"[""A"", ""O_{2}""]"
5,"""point""","""O'""","""Midpoint(O_{2}, A)""","""O' = (3.9, 0)""",3,"[""A"", ""O_{2}""]"
6,"""circle""","""p""","""Circle(O', A)""","""p: (x - 3.9)² + (y + 0)² = 36.…",3,"[""A"", ""O'"", ""O_{2}""]"
7,"""point""","""G""","""Intersect(c, p)""","""G = (2.4, 5.8)""",1,"[""A"", ""B"", … ""p""]"
8,"""point""","""F_{2}""","""Intersect(c, p)""","""F_{2} = (2.5, -5.9)""",1,"[""A"", ""B"", … ""p""]"


In [65]:
# find root
[node for node, degree in p.G.in_degree() if degree == 0]

['O_{2}', 'A']

In [67]:
p.roots, p.leaves

(['O_{2}', 'A'],
 ['n',
  'β',
  'γ',
  'a',
  'c_1',
  'g_1',
  'a_1',
  'c_2',
  'i_1',
  'k',
  'm',
  'δ',
  'ε',
  'q',
  'H',
  'J',
  'ζ',
  'η',
  'θ',
  's',
  'l_1',
  'm_1',
  'n_1',
  'q_1',
  'r_1',
  'd',
  'o',
  'l_2',
  'α',
  'f',
  'i',
  'c_3',
  'i_2',
  'a_2',
  'h',
  'g',
  'f_1',
  's_1',
  'd_1',
  'e_1',
  'F',
  'f_2',
  'e_2',
  'o_3',
  'o_{2}',
  'b_1',
  'g_2',
  'h_2',
  'j_2',
  's_2',
  't_2',
  'b_2',
  'T',
  'ι',
  'κ',
  'λ',
  'μ',
  'f_3',
  'g_3'])

## Utilities and analysis

These cells provide small utilities for working with constructions and snapshots produced by `ggblab` and `ggblab_extra`.

What these do:

- Save or export snapshots (base64 zip) to disk for later inspection.
- Decode base64 snapshots and inspect the embedded `.ggb` zip contents (including `geogebra.xml`).
- Provide graph-analysis helpers (e.g., longest-path, shortest-path length) to inspect construction dependency graphs built by `ConstructionTreeParser`.
- Offer lightweight helpers for exploring `ggb` applet internals (comm logs, pending messages, etc.).

Important: these cells focus on analysis and helper utilities — they do not perform automatic cleanup or remove application state.

In [24]:
def get_longest_path_to_node(G, target_node):
    roots = [node for node, degree in G.in_degree() if degree == 0]
    longest_path = []
    max_len = 0
    for root in roots:
        for path in nx.all_simple_paths(G, root, target_node):
            if len(path) > max_len:
                max_len = len(path)
                longest_path = path
    return longest_path

In [25]:
path = get_longest_path_to_node(p.G, 'I')
path

['O_{2}', 'B', 'c', 'j', 'r', 'O_{1}', 't', 'F_{1}', 'h_1', 'k_1', 'I']

In [26]:
depths = nx.shortest_path_length(p.G, source='A')
depths

{'A': 0,
 'B': 1,
 'n': 1,
 "O'": 1,
 'p': 1,
 'j': 1,
 'l': 1,
 'β': 1,
 'γ': 1,
 't1': 1,
 'a': 1,
 'g_1': 1,
 't2': 1,
 'a_1': 1,
 'i_1': 1,
 'k': 1,
 'm': 1,
 'δ': 1,
 'ε': 1,
 'e': 1,
 'ζ': 1,
 'η': 1,
 'θ': 1,
 'q_1': 1,
 'r_1': 1,
 't4': 1,
 'c_3': 1,
 'i_2': 1,
 'g': 1,
 'f_1': 1,
 's_1': 1,
 't_1': 1,
 'c': 2,
 'G': 2,
 'F_{2}': 2,
 'q': 2,
 'r': 2,
 'C_{1}': 2,
 'H': 2,
 's': 2,
 "P'": 2,
 'J': 2,
 'K': 2,
 'P': 2,
 't': 2,
 'F_{1}': 2,
 'C': 2,
 'f': 2,
 'e_1': 2,
 "C'": 2,
 'c_1': 2,
 'c_2': 2,
 'A_{1}': 2,
 'a_2': 2,
 'I': 3,
 'b_1': 3,
 'g_2': 3,
 'k_2': 3,
 'p_2': 3,
 'n_1': 3,
 'O': 3,
 'i': 3,
 'h': 3,
 'd_1': 3,
 'O_{1}': 3,
 'j_1': 3,
 'k_1': 3,
 'l_1': 3,
 'm_1': 3,
 'E': 3,
 'F': 3,
 'h_1': 3,
 'd_2': 3,
 "O''": 3,
 'h_2': 3,
 'j_2': 3,
 'm_2': 4,
 'n_2': 4,
 'q_2': 4,
 'r_2': 4,
 'b': 4,
 'O_3': 4,
 't3': 4,
 'd': 4,
 'o': 4,
 'α': 4,
 'Q': 4,
 'R': 4,
 'f_2': 4,
 't5': 4,
 'o_3': 4,
 'o_{2}': 4,
 'p_1': 5,
 'e_2': 5,
 'l_2': 5,
 'D': 6}

In [27]:
depths = nx.shortest_path_length(p.G2, source='A')
depths

{'A': 0,
 'B': 1,
 "O'": 1,
 'e': 2,
 'c': 2,
 'p': 2,
 'l': 3,
 'j': 3,
 'G': 3,
 'F_{2}': 3,
 'r': 4,
 'O_{1}': 5,
 't': 6,
 'b': 6,
 'O_3': 6,
 'F_{1}': 7,
 "P'": 7,
 'p_1': 7,
 'h_1': 8,
 't_1': 8,
 'C': 8,
 'E': 8,
 'D': 8,
 'j_1': 9,
 "C'": 9}

In [28]:
prev = set(list(zip(*(list(p.G.in_edges('R')) + list(p.G.in_edges('S')))))[0])
prev

{'h_1', 'k_1'}

In [29]:
[(n, depths[n]) for n in depths if n in prev]

[('h_1', 8)]

In [30]:
nx.write_network_text(p.roots_to_targets_with_containers('R'))

╟── A (point)
╎   ├─╼ B (point) ╾ O_{2} (point)
╎   │   └─╼ c (circle) ╾ O_{2} (point)
╎   │       ├─╼ j (line) ╾ A (point)
╎   │       │   ├─╼ r (line) ╾ l (line)
╎   │       │   │   └─╼ O_{1} (point)
╎   │       │   │       ├─╼ b (line) ╾ O_{2} (point)
╎   │       │   │       │   └─╼ P' (point) ╾ j (line)
╎   │       │   │       │       └─╼ k_1 (line) ╾ h_1 (circle)
╎   │       │   │       │           └─╼ R (point) ╾ h_1 (circle)
╎   │       │   │       ├─╼ t (line) ╾ l (line)
╎   │       │   │       │   └─╼ F_{1} (point) ╾ l (line)
╎   │       │   │       │       └─╼ h_1 (circle) ╾ O_{1} (point)
╎   │       │   │       │           └─╼  ...
╎   │       │   │       └─╼  ...
╎   │       │   └─╼  ...
╎   │       └─╼ l (line) ╾ A (point)
╎   │           └─╼  ...
╎   └─╼  ...
╙── O_{2} (point)
    └─╼  ...


## .

In [68]:
p.df.filter(pl.col("Layer").is_in([9]))["Sequence", "Type", "Name", "Command", "Value", "Layer", "DependsOn"]

Sequence,Type,Name,Command,Value,Layer,DependsOn
u32,str,str,str,str,u32,list[str]
0,"""point""","""O_{2}""",null,"""O_{2} = (-2.1, -0.1)""",9,[]
2,"""point""","""B""","""Point(Line(O_{2}, A))""","""B = (-9.6, -0.2)""",9,"[""A"", ""O_{2}""]"
3,"""circle""","""c""","""Circle(O_{2}, B)""","""c: (x + 2.1)² + (y + 0.1)² = 5…",9,"[""A"", ""B"", ""O_{2}""]"
35,"""point""","""O_{1}""","""Point(r)""","""O_{1} = (10, -4.8)""",9,"[""A"", ""B"", … ""r""]"
43,"""circle""","""h_1""","""Circle(O_{1}, F_{1})""","""h_1: (x - 10)² + (y + 4.8)² = …",9,"[""A"", ""B"", … ""t""]"


In [197]:
's' in p.leaves

True

In [131]:
nx.descendants(p.G, 'j')

{'A_{1}',
 'C',
 "C'",
 'C_{1}',
 'D',
 'E',
 'F',
 'F_{1}',
 'H',
 'I',
 'L',
 'M',
 'N',
 'O',
 "O''",
 'O_3',
 'O_{1}',
 'P',
 "P'",
 'Q',
 'R',
 'S',
 'T',
 'a_2',
 'b',
 'b_1',
 'b_2',
 'c_3',
 'd',
 'd_1',
 'd_2',
 'e_2',
 'f',
 'f_2',
 'f_3',
 'g',
 'g_2',
 'g_3',
 'h',
 'h_1',
 'h_2',
 'i',
 'i_2',
 'j_1',
 'j_2',
 'k_1',
 'k_2',
 'l_1',
 'l_2',
 'm_1',
 'm_2',
 'n_2',
 'o',
 'o_3',
 'o_{2}',
 'p_1',
 'p_2',
 'q',
 'q_1',
 'q_2',
 'r',
 'r_2',
 's',
 's_2',
 't',
 't3',
 't4',
 't5',
 't_1',
 't_2',
 'α',
 'ζ',
 'η',
 'θ',
 'ι',
 'κ',
 'λ',
 'μ'}

In [33]:
await ggb.function("setVisible", ["C", True])

In [34]:
p.df["DependsOn"][42].to_list()

['A', 'B', 'O_{1}', 'O_{2}', 'c', 'j', 'l', 'r', 't']

In [35]:
await ggb.function("setVisible", list(zip_longest(_, [], fillvalue=True)))

[None, None, None, None, None, None, None, None, None]

## store

In [211]:
ggb.file.base64_buffer = await ggb.function("getBase64")
c.save(overwrite=True)

In [209]:
from itertools import zip_longest

l0 = range(10)
await ggb.function("setLayerVisible", list(zip_longest(list(l0), [], fillvalue=False)))
l0 = [9, 0, 2, 5]
await ggb.function("setLayerVisible", list(zip_longest(list(l0), [], fillvalue=True)))

[None, None, None, None]

In [73]:
for e in nx.ancestors(p.G, 'j_1'):
    print(e, p.ft[e])

l ['Tangent', 'A', 'c']
O_{1} ['Point', 'r']
P' ['Intersect', 'b', 'j']
c ['Circle', 'O_{2}', 'B']
B ['Point', 'Line', 'O_{2}', 'A']
b ['Line', 'O_{2}', 'O_{1}']
j ['Tangent', 'A', 'c']
F_{1} ['Intersect', 't', 'l']
A []
r ['AngleBisector', 'l', 'j']
t ['PerpendicularLine', 'O_{1}', 'l']
h_1 ['Circle', 'O_{1}', 'F_{1}']
O_{2} []


In [75]:
nx.write_network_text(p.G.subgraph(nx.ancestors(p.G, 'j_1') | {'j_1'}))

╟── A (point)
╎   ├─╼ B (point) ╾ O_{2} (point)
╎   │   └─╼ c (circle) ╾ O_{2} (point)
╎   │       ├─╼ j (line) ╾ A (point)
╎   │       │   ├─╼ r (line) ╾ l (line)
╎   │       │   │   └─╼ O_{1} (point)
╎   │       │   │       ├─╼ b (line) ╾ O_{2} (point)
╎   │       │   │       │   └─╼ P' (point) ╾ j (line)
╎   │       │   │       │       └─╼ j_1 (line) ╾ h_1 (circle)
╎   │       │   │       ├─╼ t (line) ╾ l (line)
╎   │       │   │       │   └─╼ F_{1} (point) ╾ l (line)
╎   │       │   │       │       └─╼ h_1 (circle) ╾ O_{1} (point)
╎   │       │   │       │           └─╼  ...
╎   │       │   │       └─╼  ...
╎   │       │   └─╼  ...
╎   │       └─╼ l (line) ╾ A (point)
╎   │           └─╼  ...
╎   └─╼  ...
╙── O_{2} (point)
    └─╼  ...


In [84]:
s = set()
for e in ['j_1', 'k_1', 'l_1', 'm_1']:
    s |= nx.ancestors(p.G, e) | {e}
s

{'A',
 'B',
 'F_{1}',
 'O_{1}',
 'O_{2}',
 'P',
 "P'",
 'b',
 'c',
 'h_1',
 'j',
 'j_1',
 'k_1',
 'l',
 'l_1',
 'm_1',
 'r',
 't'}

In [85]:
from itertools import zip_longest

l0 = range(10)
await ggb.function("setLayerVisible", list(zip_longest(list(l0), [], fillvalue=False)))
l1 = list(s) #['n', 'r', 'O_{1}', 'O_{2}', 'c', 'h_1', 'A', 'e_1', 'F_{1}', 'F_{2}', 't', 'j', 'l'] # 'C',  'k_1', "P'", 'l_1'
await ggb.function("setVisible", list(zip_longest(list(l1), [], fillvalue=True)))

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [100]:
for e in l1:
    print(e, p.ft[e])

n ['Line', 'O_{2}', 'A']
r ['AngleBisector', 'l', 'j']
O_{1} ['Point', 'r']
O_{2} []
c ['Circle', 'O_{2}', 'B']
h_1 ['Circle', 'O_{1}', 'F_{1}']
P' ['Intersect', 'b', 'j']
A []
C ['Intersect', 'p_1', 'l', '2']
F_{1} ['Intersect', 't', 'l']
F_{2} ['Intersect', 'c', 'p']
t ['PerpendicularLine', 'O_{1}', 'l']
j ['Tangent', 'A', 'c']
k_1 ['Tangent', "P'", 'h_1']
l ['Tangent', 'A', 'c']
l_1 ['Tangent', 'P', 'h_1']


In [106]:
nx.write_network_text(p.G.subgraph(l1))

╟── O_{2} (point)
╎   ├─╼ c (circle)
╎   │   ├─╼ F_{2} (point)
╎   │   │   └─╼ e_1 (line) ╾ l (line)
╎   │   ├─╼ j (line) ╾ A (point)
╎   │   │   └─╼ r (line) ╾ l (line)
╎   │   │       └─╼ O_{1} (point)
╎   │   │           ├─╼ t (line) ╾ l (line)
╎   │   │           │   └─╼ F_{1} (point) ╾ l (line)
╎   │   │           │       └─╼ h_1 (circle) ╾ O_{1} (point)
╎   │   │           │           └─╼ l_1 (line)
╎   │   │           └─╼  ...
╎   │   └─╼ l (line) ╾ A (point)
╎   │       ├─╼ C (point)
╎   │       └─╼  ...
╎   └─╼ n (line) ╾ A (point)
╙── A (point)
    └─╼  ...


In [36]:
p.ft['b']

['Line', 'O_{2}', 'O_{1}']

In [91]:
p.fbd('c')

{'A', 'B', 'O_{2}'}

In [110]:
await ggb.command("Circle(F_{2}, Distance(O_{1}, A))")

't_1'

In [112]:
await ggb.command("C' = Intersect[l, t_1, 2]")

"C'"

In [148]:
await ggb.command("Intersect(l_1, h_1)")

'U'

In [138]:
await ggb.command("O'' = Midpoint(C', O_{1})")

"O''"

In [146]:
await ggb.command("Line(C', O_{1})")

'j_2'

In [88]:
await ggb.command("Circle(V, Radius(h_1))")

'd_3'

In [149]:
await ggb.command("Circle(O_{2}, Radius(c) + Radius(h_1))")

'p_2'

In [150]:
await ggb.command("Tangent(O_{1}, k_2)")

'm_2,n_2'

In [151]:
await ggb.command("Tangent(O_{1}, p_2)")

'q_2,r_2'

In [40]:
await ggb.command("Intersect[k_2, n_2]")

'S'

In [87]:
await ggb.command("Intersect[c, b]")

'U,V'

In [151]:
await ggb.command("Angle[O_{1}, C, O_{2}]")

'ν'

In [150]:
await ggb.command("AngleBisector[k_1, m_1]")

'h_3,i_3'

In [97]:
nx.descendants(p.G, 'e')

{'A_{1}', 'C_{1}', 'H', 'J', 'K', 'ζ', 'η'}

In [98]:
p.ft['ζ']

['Angle', 'A_{1}', 'A', 'K']